# Sports RAG — Data Collection & Chunking

Install dependencies

In [2]:
!pip install wikipedia-api wikipedia langchain langchain-text-splitters \
             sentence-transformers pinecone tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 16.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.9 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=e00e18855054d60a953ba095186484e2d2d1536a8501d74349963b0dc5fbd287
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframe

## Cell 2 — Imports

In [3]:
import wikipedia
import json
import re
import time
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

wikipedia.set_lang("en")
print("All imports OK")

All imports OK


## Cell 3 — Topic list (edit freely)
These cover football, cricket, basketball, tennis and general sports history.
Feel free to add or remove topics — aim for 60+ to guarantee 500+ chunks.

In [4]:
TOPICS = [
    # --- Football / Soccer: Tournaments & competitions ---
    "FIFA World Cup",
    "2022 FIFA World Cup",
    "2018 FIFA World Cup",
    "2014 FIFA World Cup",
    "UEFA Champions League",
    "UEFA Europa League",
    "UEFA Conference League",
    "UEFA Super Cup",
    "UEFA European Championship",
    "Copa America",
    "Copa Libertadores",
    "Copa del Rey",
    "FA Cup",
    "EFL Cup",
    "Premier League",
    "La Liga",
    "Serie A",
    "Bundesliga",
    "Ligue 1",
    "Primeira Liga",
    "Eredivisie",
    "Major League Soccer",
    "Saudi Pro League",
    "AFC Champions League",

    # --- Football / Soccer: Clubs ---
    "Real Madrid CF",
    "FC Barcelona",
    "Manchester United F.C.",
    "Manchester City F.C.",
    "Liverpool F.C.",
    "Chelsea F.C.",
    "Arsenal F.C.",
    "Tottenham Hotspur F.C.",
    "Bayern Munich",
    "Borussia Dortmund",
    "Juventus FC",
    "Inter Milan",
    "A.C. Milan",
    "Napoli",
    "Paris Saint-Germain F.C.",
    "Atletico Madrid",
    "Ajax Amsterdam",
    "Benfica",
    "FC Porto",

    # --- Football / Soccer: Players ---
    "Lionel Messi",
    "Cristiano Ronaldo",
    "Neymar",
    "Kylian Mbappe",
    "Erling Haaland",
    "Kevin De Bruyne",
    "Luka Modric",
    "Karim Benzema",
    "Mohamed Salah",
    "Robert Lewandowski",
    "Virgil van Dijk",
    "Jude Bellingham",
    "Vinicius Junior",
    "Harry Kane",
    "Zinedine Zidane",
    "Ronaldinho",
    "Ronaldo (Brazilian footballer)",
    "Pele",
    "Diego Maradona",
    "Johan Cruyff",

    # --- Football / Soccer: International teams & history ---
    "Brazil national football team",
    "Argentina national football team",
    "France national football team",
    "Germany national football team",
    "Spain national football team",
    "England national football team",
    "Portugal national football team",
    "Italy national football team",
    "Netherlands national football team",
    "Croatia national football team",
    "History of association football",
    "Laws of the Game (association football)",
    "VAR (association football)",
    "FIFA",
    "UEFA",
    "Ballon d'Or",
    "FIFA The Best Men's Player",
]

print(f"Total topics defined: {len(TOPICS)}")

Total topics defined: 80


## Cell 4 — Fetch Wikipedia articles
This may take 3-5 minutes. Failed topics are skipped automatically.

In [5]:
def clean_text(text: str) -> str:
    """Remove citation markers, excess whitespace, and short lines."""
    text = re.sub(r"=={1,3}.*?=={1,3}", "", text)   # section headers
    text = re.sub(r"\[\d+\]", "", text)              # [1] [2] citations
    text = re.sub(r"\n{3,}", "\n\n", text)           # triple+ newlines
    text = re.sub(r"[ \t]+", " ", text)              # multiple spaces
    return text.strip()


raw_docs = []
failed  = []

for topic in tqdm(TOPICS, desc="Fetching articles"):
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        content = clean_text(page.content)
        if len(content) > 300:          # skip stubs
            raw_docs.append({
                "title"  : page.title,
                "url"    : page.url,
                "content": content
            })
    except wikipedia.exceptions.DisambiguationError as e:
        # try the first suggested option
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
            content = clean_text(page.content)
            raw_docs.append({"title": page.title, "url": page.url, "content": content})
        except:
            failed.append(topic)
    except Exception as ex:
        failed.append(topic)
    time.sleep(0.3)   # be polite to Wikipedia

print(f"\nSuccessfully fetched : {len(raw_docs)} articles")
print(f"Failed / skipped     : {len(failed)} -> {failed}")

# Save raw backup
with open("corpus_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_docs, f, ensure_ascii=False, indent=2)
print("corpus_raw.json saved.")

Fetching articles: 100%|██████████| 80/80 [01:10<00:00,  1.13it/s]


Successfully fetched : 79 articles
Failed / skipped     : 1 -> ["FIFA The Best Men's Player"]
corpus_raw.json saved.


## Cell 4b — Preprocessing
Runs targeted, non-aggressive cleaning on the fetched articles before chunking.
Goals:
- Remove IPA / phonetic pronunciation blocks that add no factual value
- Strip Wikipedia hatnote boilerplate ("For other uses…", "Not to be confused with…")
- Drop lines that are pure numbers (leaked table rows)
- Normalise Unicode to NFC canonical form (also fixes non-ASCII Pinecone ID issues)

The cleaned corpus is stored in `docs` and used for all downstream steps.

In [6]:
# --- Cell 4b: Preprocessing — targeted noise removal for RAG grounding ---
import unicodedata

def preprocess_doc(doc: dict) -> dict:
    """
    Light-touch cleanup to improve retrieval grounding quality.
    Operates on a single document dict {title, url, content}.
    Does NOT remove factual content — only Wikipedia-specific noise.
    """
    text  = doc["content"]
    title = doc["title"]

    # 1. Remove parenthetical blocks containing non-ASCII glyphs
    #    Catches IPA: ( FED-ər-ər; Swiss …)  ([ˈbʊndəsˌliːɡa])
    text = re.sub(r'\([^)]*[\u0080-\uFFFF][^)]*\)', ' ', text)
    text = re.sub(r'\[[^\]]*[\u0080-\uFFFF][^\]]*\]', ' ', text)

    # 2. Strip Wikipedia hatnote / disambiguation boilerplate lines
    hatnote_pats = [
        r'^This article (is about|covers|describes)\b.*$',
        r'^For (other uses|the [\w ]+),? see\b.*$',
        r'^Not to be confused with\b.*$',
        r'^See also\s*:.*$',
        r'^Further information\s*:.*$',
        r'^This (section|article) (needs|requires|may need)\b.*$',
    ]
    for pat in hatnote_pats:
        text = re.sub(pat, '', text, flags=re.MULTILINE | re.IGNORECASE)

    # 3. Drop lines that are purely numeric (leaked stat-table rows)
    #    e.g. "     1       2       3"  or "128,456"
    text = re.sub(r'^\s*[\d,\.\s]+\s*$', '', text, flags=re.MULTILINE)

    # 4. Collapse excess whitespace left behind
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = text.strip()

    # 5. Unicode NFC normalisation — canonical form, no aggressive transliteration
    #    Keeps accented names intact in content while making metadata safe downstream
    text  = unicodedata.normalize("NFC", text)
    title = unicodedata.normalize("NFC", title)

    return {**doc, "content": text, "title": title}


# Apply to every fetched document
docs = [preprocess_doc(d) for d in raw_docs]

# ── Reporting ──────────────────────────────────────────────────────────────
orig_chars  = sum(len(d["content"]) for d in raw_docs)
clean_chars = sum(len(d["content"]) for d in docs)
removed     = orig_chars - clean_chars
removed_pct = removed / orig_chars * 100

print(f"Documents processed  : {len(docs)}")
print(f"Chars before cleaning: {orig_chars:,}")
print(f"Chars after cleaning : {clean_chars:,}")
print(f"Noise removed        : {removed:,}  ({removed_pct:.1f}%)")
print()

# Quick sample diff for one article
sample_before = raw_docs[7]["content"][:600]
sample_after  = docs[7]["content"][:600]
if sample_before != sample_after:
    print(f"[Sample diff — {docs[7]['title']}]")
    print("BEFORE:", repr(sample_before[:200]))
    print("AFTER :", repr(sample_after[:200]))
else:
    print(f"[{docs[7]['title']}] — no change needed (already clean)")


Documents processed  : 79
Chars before cleaning: 3,254,698
Chars after cleaning : 3,236,009
Noise removed        : 18,689  (0.6%)

[UEFA Super Cup] — no change needed (already clean)


In [7]:
# --- Cell: Inspect fetched documents (all + samples) ---
import pandas as pd

# Prefer in-memory docs; fallback to saved file if needed
if "docs" in globals() and isinstance(docs, list):
    docs_for_view = docs
else:
    with open("corpus_raw.json", "r", encoding="utf-8") as f:
        docs_for_view = json.load(f)

if not docs_for_view:
    print("No documents found. Run the fetch cell first.")
else:
    # Show all documents in a compact table
    rows = []
    for d in docs_for_view:
        text = d.get("content", "")
        rows.append({
            "title": d.get("title", ""),
            "url": d.get("url", ""),
            "chars": len(text),
            "words": len(text.split()),
            "preview": text[:160].replace("\n", " ") + ("..." if len(text) > 160 else "")
        })

    docs_df = pd.DataFrame(rows).sort_values("words", ascending=False).reset_index(drop=True)
    print(f"Fetched documents: {len(docs_df)}")
    display(docs_df)

    # Also print readable samples (useful when table output is truncated)
    sample_n = min(10, len(docs_for_view))
    print(f"\nShowing {sample_n} sample documents:\n")
    for i, d in enumerate(docs_for_view[:sample_n], 1):
        text = d.get("content", "")
        print(f"[{i}] {d.get('title', '')}")
        print(f"URL: {d.get('url', '')}")
        print(f"Words: {len(text.split())}")
        print(f"Preview: {text[:300].replace(chr(10), ' ')}")
        print("-" * 90)

Fetched documents: 79


,title,url,chars,words,preview
0,Johan Cruyff,https://en.wikipedia.org/wiki/Johan_Cruyff,116372,18898,"Hendrik Johannes Cruijff , known as Johan Cruy..."
1,Neymar,https://en.wikipedia.org/wiki/Neymar,110029,18859,"Neymar da Silva Santos Júnior , simply known a..."
2,Diego Maradona,https://en.wikipedia.org/wiki/Diego_Maradona,80904,13589,Diego Armando Maradona was an Argentine profes...
3,Mohamed Salah,https://en.wikipedia.org/wiki/Mohamed_Salah,75848,13054,Mohamed Salah Hamed Mahrous Ghaly (born 15 Jun...
4,Real Madrid CF,https://en.wikipedia.org/wiki/Real_Madrid_CF,76165,12667,"Real Madrid Club de Fútbol , commonly referred..."
...,...,...,...,...,...
74,UEFA Conference League,https://en.wikipedia.org/wiki/UEFA_Conference_...,12388,1981,"The UEFA Conference League (UECL), usually kno..."
75,AFC Champions League Elite,https://en.wikipedia.org/wiki/AFC_Champions_Le...,11187,1837,The AFC Champions League Elite (abbreviated as...
76,Copa del Rey,https://en.wikipedia.org/wiki/Copa_del_Rey,9815,1644,The Campeonato de España–Copa de Su Majestad e...
77,Ballon d'Or,https://en.wikipedia.org/wiki/Ballon_d%27Or,9243,1505,The Ballon d'Or is an annual association footb...



Showing 10 sample documents:

[1] FIFA World Cup
URL: https://en.wikipedia.org/wiki/FIFA_World_Cup
Words: 6038
Preview: The FIFA World Cup, often called the World Cup, is an international association football competition among the senior men's national teams of the members of the Fédération Internationale de Football Association (FIFA), the sport's global governing body. The tournament has been held every four years 
------------------------------------------------------------------------------------------
[2] 2022 FIFA World Cup
URL: https://en.wikipedia.org/wiki/2022_FIFA_World_Cup
Words: 11133
Preview: The 2022 FIFA World Cup was the 22nd FIFA World Cup, the quadrennial world championship for national football teams organised by FIFA. It took place in Qatar from 20 November to 18 December 2022, after the country was awarded the hosting rights in 2010. It was the first World Cup to be held in the M
-------------------------------------------------------------------------------------

## Cell 5 — Chunk the articles (three strategies)
We produce **Fixed**, **Recursive**, and **Semantic** chunks for the ablation study.

| Strategy | Description |
|---|---|
| Fixed | 500-char windows, no overlap |
| Recursive | 500-char windows, 50-char overlap, hierarchy-aware separators |
| **Semantic** | Sentence-embedding cosine similarity breakpoints — topic-aware splits |

The Recursive chunks are written to `chunks.json` (production default).
Swap to `chunks_semantic.json` to use semantic chunking in the embed cell.

In [8]:
import numpy as np
# --- Strategy A: Fixed-size chunking ---
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 150,       
    length_function = len,
)

# --- Strategy B: Recursive chunking (production default) ---
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 100,
    length_function = len,
    separators    = ["\n\n", "\n", ". ", " ", ""],
    # separators = ["\n\n", "\n", ". ", "? ", "! ", " ", ""]
)


def _ascii_safe_id(text: str) -> str:
    """Return a Pinecone-safe ASCII token for vector IDs."""
    import unicodedata
    ascii_text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    ascii_text = re.sub(r"\s+", "_", ascii_text.strip())
    ascii_text = re.sub(r"[^A-Za-z0-9_\-]", "", ascii_text)
    return ascii_text or "untitled"


def make_chunks(docs, splitter, strategy_name):
    chunks = []
    for doc in docs:
        parts = splitter.split_text(doc["content"])
        safe_source = _ascii_safe_id(doc["title"])
        for i, part in enumerate(parts):
            part = part.strip()
            if len(part) < 50:
                continue
            chunks.append({
                "id"      : f"{strategy_name}_{safe_source}_{i}",
                "text"    : part,
                "source"  : doc["title"],
                "url"     : doc["url"],
                "strategy": strategy_name,
                "chunk_no": i
            })
    return chunks


def make_semantic_chunks(docs, strategy_name="semantic", max_chars=1000, min_chars=250,
                         similarity_break=0.32, model_name="all-MiniLM-L6-v2"):
    """
    Semantic-ish chunking:
    - split into sentences
    - break chunk if topical shift appears (low adjacent sentence similarity)
    - also enforce max chunk length
    """
    sem_model = SentenceTransformer(model_name)
    chunks = []

    for doc in tqdm(docs, desc="Semantic chunking"):
        text = doc["content"].strip()
        if not text:
            continue

        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
        if not sentences:
            continue

        embeddings = sem_model.encode(sentences, show_progress_bar=False, normalize_embeddings=True)

        safe_source = _ascii_safe_id(doc["title"])
        current = [sentences[0]]
        current_len = len(sentences[0])
        out_idx = 0

        for i in range(1, len(sentences)):
            sim = float(np.dot(embeddings[i - 1], embeddings[i]))
            sent = sentences[i]
            sent_len = len(sent)

            should_break = (
                (current_len >= min_chars and sim < similarity_break) or
                (current_len + 1 + sent_len > max_chars)
            )

            if should_break:
                chunk_text = " ".join(current).strip()
                if len(chunk_text) >= 50:
                    chunks.append({
                        "id"      : f"{strategy_name}_{safe_source}_{out_idx}",
                        "text"    : chunk_text,
                        "source"  : doc["title"],
                        "url"     : doc["url"],
                        "strategy": strategy_name,
                        "chunk_no": out_idx
                    })
                    out_idx += 1
                current = [sent]
                current_len = sent_len
            else:
                current.append(sent)
                current_len += 1 + sent_len

        final_chunk = " ".join(current).strip()
        if len(final_chunk) >= 50:
            chunks.append({
                "id"      : f"{strategy_name}_{safe_source}_{out_idx}",
                "text"    : final_chunk,
                "source"  : doc["title"],
                "url"     : doc["url"],
                "strategy": strategy_name,
                "chunk_no": out_idx
            })

    return chunks


source_docs = docs if "docs" in globals() else raw_docs
fixed_chunks     = make_chunks(source_docs, fixed_splitter, "fixed")
recursive_chunks = make_chunks(source_docs, recursive_splitter, "recursive")
semantic_chunks  = make_semantic_chunks(source_docs, strategy_name="semantic")

print(f"Fixed chunks     : {len(fixed_chunks)}")
print(f"Recursive chunks : {len(recursive_chunks)}")
print(f"Semantic chunks  : {len(semantic_chunks)}")

with open("chunks_fixed.json", "w", encoding="utf-8") as f:
    json.dump(fixed_chunks, f, ensure_ascii=False, indent=2)

with open("chunks_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)

with open("chunks_semantic.json", "w", encoding="utf-8") as f:
    json.dump(semantic_chunks, f, ensure_ascii=False, indent=2)

# Default app corpus (can be swapped later)
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)

print("\nSaved: chunks.json, chunks_fixed.json, chunks_recursive.json, chunks_semantic.json")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic chunking: 100%|██████████| 79/79 [00:12<00:00,  6.17it/s]


Fixed chunks     : 4876
Recursive chunks : 4864
Semantic chunks  : 4997

Saved: chunks.json, chunks_fixed.json, chunks_recursive.json, chunks_semantic.json


## Cell 6 — Quick sanity check

In [9]:
import json

with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks loaded : {len(chunks)}")
print(f"Avg chunk length    : {sum(len(c['text']) for c in chunks) // len(chunks)} chars")
print(f"Unique sources      : {len(set(c['source'] for c in chunks))}")
print()
print("--- Sample chunk ---")
sample = chunks[50]
print(f"ID     : {sample['id']}")
print(f"Source : {sample['source']}")
print(f"Text   : {sample['text'][:300]}...")

Total chunks loaded : 4864
Avg chunk length    : 667 chars
Unique sources      : 79

--- Sample chunk ---
ID     : recursive_FIFA_World_Cup_50
Source : FIFA World Cup
Text   : the Golden Ball (named for its sponsor "Adidas Golden Ball") for best player, first awarded in 1982;
the Golden Boot (named for its sponsor "Adidas Golden Boot", formerly known as the "adidas Golden Shoe" from 1982 to 2006) for top goalscorer, first awarded in 1982;
the Golden Glove (named for its s...


## Cell 7 — Embed chunks & upsert to Pinecone


In [10]:
import os
import torch

PINECONE_API_KEY = ""  # insert your Pinecone API key
INDEX_NAME       = "sports-rag"
BATCH_SIZE       = 100

# Better GPU-friendly alternatives for Kaggle:
# - BAAI/bge-small-en-v1.5  (fast, strong, dim=384)
# - intfloat/e5-base-v2     (stronger, dim=768)
# - BAAI/bge-base-en-v1.5   (strong, dim=768)
EMBEDDING_MODEL  = "BAAI/bge-small-en-v1.5"

if not PINECONE_API_KEY.strip():
    raise ValueError("PINECONE_API_KEY is empty. Add your key before running this cell.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading embedding model on {DEVICE}: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print("Model loaded.")


def load_chunks(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run the chunking cell first.")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


chunk_sets = {
    "fixed": load_chunks("chunks_fixed.json"),
    "recursive": load_chunks("chunks_recursive.json"),
    "semantic": load_chunks("chunks_semantic.json"),
}

for name, chunks in chunk_sets.items():
    bad_ids = [c["id"] for c in chunks if not str(c.get("id", "")).isascii()]
    if bad_ids:
        raise ValueError(f"{name}: found non-ASCII ID. Example: {bad_ids[0]}")

# Infer embedding dimension from model output
sample_vec = model.encode(["dimension probe"], show_progress_bar=False)
EMBEDDING_DIM = int(len(sample_vec[0]))
print(f"Embedding dimension: {EMBEDDING_DIM}")

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create/validate index
existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    idx_info = pc.describe_index(INDEX_NAME)
    existing_dim = idx_info.dimension if hasattr(idx_info, "dimension") else idx_info["dimension"]
    if int(existing_dim) != EMBEDDING_DIM:
        raise ValueError(
            f"Index '{INDEX_NAME}' has dim={existing_dim}, but model needs dim={EMBEDDING_DIM}. "
            "Create a new index name or recreate this one."
        )
    print(f"Index '{INDEX_NAME}' already exists.")

index = pc.Index(INDEX_NAME)


def upsert_chunks(chunks, namespace):
    print(f"\nUpserting namespace='{namespace}' | chunks={len(chunks)}")
    for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc=f"Upserting {namespace}"):
        batch = chunks[i:i + BATCH_SIZE]
        texts = [c["text"] for c in batch]
        embeddings = model.encode(texts, show_progress_bar=False).tolist()
        vectors = [
            (
                c["id"],
                emb,
                {"text": c["text"], "source": c["source"], "url": c["url"], "strategy": c.get("strategy", namespace)}
            )
            for c, emb in zip(batch, embeddings)
        ]
        index.upsert(vectors=vectors, namespace=namespace)


# Upsert thrice: once per chunking strategy
for strategy_name, strategy_chunks in chunk_sets.items():
    upsert_chunks(strategy_chunks, namespace=strategy_name)

print("\nAll chunk sets embedded and upserted!")
stats = index.describe_index_stats()
print("Index stats:")
print(stats)

Loading embedding model on cuda: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...


Upserting namespace='recursive' | chunks=4864


Upserting recursive: 100%|██████████| 49/49 [00:37<00:00,  1.30it/s]



Upserting namespace='semantic' | chunks=4997


Upserting semantic: 100%|██████████| 50/50 [00:41<00:00,  1.19it/s]


All chunk sets embedded and upserted!
Index stats:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '247',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 26 Mar 2026 18:11:09 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'fixed': {'vector_count': 4876},
                'recursive': {'vector_count': 4864},
                'semantic': {'vector_count': 4997}},
 'storageFullness': 0.0,
 'total_vector_count': 14737,
 'vector_type': 'dense'}


## Cell 8 — Verify Pinecone with a test query

In [11]:
test_query = "Who won the FIFA World Cup in 2022?"

# BGE models generally perform better for retrieval with an explicit query instruction.
query_for_embedding = f"Represent this sentence for searching relevant passages: {test_query}"
query_vec = model.encode(query_for_embedding, normalize_embeddings=True).tolist()

def show_results(namespace, top_k=5):
    results = index.query(
        vector=query_vec,
        top_k=top_k,
        include_metadata=True,
        namespace=namespace
    )
    print(f"\n=== Namespace: {namespace} | Query: {test_query} ===")
    matches = results.get("matches", [])
    if not matches:
        print("No matches returned.")
        return

    for i, match in enumerate(matches, 1):
        md = match.get("metadata", {})
        text = md.get("text", "")
        source = md.get("source", "Unknown")
        print(f"--- Result {i} (score: {match.get('score', 0):.4f}) ---")
        print(f"Source : {source}")
        print(f"Text   : {text[:5000]}...")
        print()

# Compare retrieval quality side-by-side per chunking strategy
for ns in ["fixed", "recursive", "semantic"]:
    show_results(ns, top_k=5)

# Optional: inspect what _default_ contains (likely older mixed data)
print("\nTip: If _default_ has stale vectors, avoid querying it for evaluation.")


=== Namespace: fixed | Query: Who won the FIFA World Cup in 2022? ===
--- Result 1 (score: 0.8567) ---
Source : 2022 FIFA World Cup
Text   : The 2022 FIFA World Cup was the 22nd FIFA World Cup, the quadrennial world championship for national football teams organised by FIFA. It took place in Qatar from 20 November to 18 December 2022, after the country was awarded the hosting rights in 2010. It was the first World Cup to be held in the Middle East and the Arabian Peninsula, and the second in an Asian country after the 2002 tournament in South Korea and Japan....

--- Result 2 (score: 0.8260) ---
Source : FIFA World Cup
Text   : As of the 2022 World Cup, 22 final tournaments have been held since the event's inception in 1930, and a total of 80 national teams have competed. The trophy has been won by eight national teams. With five wins, Brazil is the only team to have played in every tournament. The other World Cup winners are Germany and Italy, with four titles each; Argentina, with t

## Done!
Your output files:
| File | Purpose |
|---|---|
| `corpus_raw.json` | Raw Wikipedia articles (backup) |
| `chunks.json` | Recursive chunks → upload to HF Space |
| `chunks_fixed.json` | Fixed chunks → for ablation study |
| `chunks_recursive.json` | Same as chunks.json (explicit copy) |

**Next step:** Upload `chunks.json` to your HF Space repo and build the BM25 + semantic search pipeline.